In [ ]:
import pandas as pd
import os

# =========================
# 1. LOAD DATASET
# =========================
df1 = pd.read_csv("dataset1.csv")   # ganti dengan path dataset 1
df2 = pd.read_csv("dataset2.csv")   # ganti dengan path dataset 2

print("Ukuran awal:")
print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)

# =========================
# 2. AMBIL 2500 DATA PER DATASET
# =========================
df1_sample = df1.sample(n=min(2500, len(df1)), random_state=42)
df2_sample = df2.sample(n=min(2500, len(df2)), random_state=42)

print("\nSetelah sampling:")
print("Dataset 1:", df1_sample.shape)
print("Dataset 2:", df2_sample.shape)

# =========================
# 3. SAMAKAN KOLOM
# =========================
common_cols = [
    "title",
    "description",
    "requirements",
    "company_profile",
    "location",
    "salary_range",
    "employment_type",
    "industry",
    "benefits",
    "fraudulent"
]

df1_clean = df1_sample[common_cols]
df2_clean = df2_sample[common_cols]

# =========================
# 4. GABUNGKAN DATASET
# =========================
df_merged = pd.concat(
    [df1_clean, df2_clean],
    ignore_index=True
)

# =========================
# 5. SHUFFLE DATA
# =========================
df_merged = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nUkuran akhir dataset gabungan:")
print(df_merged.shape)  # harusnya (5000, 10)

# =========================
# 6. HANDLE MISSING VALUE
# =========================
df_merged = df_merged.fillna("")

# =========================
# 7. CEK DISTRIBUSI LABEL
# =========================
print("\nDistribusi label fraudulent:")
print(df_merged["fraudulent"].value_counts())

# =========================
# 8. SIMPAN KE CSV
# =========================
output_path = "dataset_job_5000.csv"

df_merged.to_csv(output_path, index=False)

print(f"\n✅ Dataset berhasil disimpan ke: {output_path}")


Ukuran awal:
Dataset 1: (10000, 10)
Dataset 2: (17880, 18)

Setelah sampling:
Dataset 1: (2500, 10)
Dataset 2: (2500, 18)

Ukuran akhir dataset gabungan:
(5000, 10)

Distribusi label fraudulent:
fraudulent
1    2633
0    2367
Name: count, dtype: int64

✅ Dataset berhasil disimpan ke: dataset_job_5000.csv


# **LSTM**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


In [ ]:
df = pd.read_csv("dataset_job_5000.csv")

print(df.shape)
df.head()

(5000, 10)


,title,description,requirements,company_profile,location,salary_range,employment_type,industry,benefits,fraudulent
0,Health and safety adviser,Lead unit better computer assume majority. Ear...,"Basic knowledge in community, no degree requir...",Williams PLC - Established 1976.,Martinshire,$59176-$71806,Part-Time,Finance,Remote work opportunities,1
1,Graduates: English Teacher Overseas,"Play with kids, get paid for it :-)Love travel...",University degree required. TEFL / TESOL / CEL...,We help teachers get safe &amp; secure jobs ab...,"ZA, WC, Cape Town",NaN,Contract,Education Management,See job description,0
2,Marketing Professional,Valor Services is searching for an exceptional...,QUALIFICATIONS REQUIRED:A bachelor’s degree in...,Valor Services provides Workforce Solutions th...,"US, PA,",NaN,Full-time,Oil & Energy,NaN,0
3,"Surveyor, building",Service social eight buy. Earn $5000/week! Imm...,"Basic knowledge in occur, no degree required. ...","Cox, Byrd and Rosario - Established 1972.",Toddview,$44289-$111142,Internship,Automotive,Sign-on bonus,1
4,Ergonomist,Assume then late kind little. Earn $5000/week!...,"Basic knowledge in new, no degree required. Fl...",Gallegos Group - Established 1970.,West Kimberly,$32645-$100782,Part-Time,IT,Flexible hours,1


In [ ]:
df = df.fillna("")

df["text"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["company_profile"]
)

X = df["text"]
y = df["fraudulent"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

print(X_train_pad.shape, X_test_pad.shape)


(4000, 200) (1000, 200)


In [ ]:
model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 53s 446ms/step - accuracy: 0.8175 - loss: 0.5070 - val_accuracy: 0.8875 - val_loss: 0.3114
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 310ms/step - accuracy: 0.8887 - loss: 0.3356 - val_accuracy: 0.8875 - val_loss: 0.3151
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 41s 313ms/step - accuracy: 0.9021 - loss: 0.2809 - val_accuracy: 0.9650 - val_loss: 0.1056
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 37s 323ms/step - accuracy: 0.9433 - loss: 0.1477 - val_accuracy: 0.8875 - val_loss: 0.3016
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 34s 302ms/step - accuracy: 0.9221 - loss: 0.1919 - val_accuracy: 0.9750 - val_loss: 0.1042
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 37s 323ms/step - accuracy: 0.9851 - loss: 0.0500 - val_accuracy: 0.9800 - val_loss: 0.1052
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 36s 315ms/step - accuracy: 0.9957 - loss: 0.0194 - val_accuracy: 0.9825 - val_loss: 0.0854
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 37s 327ms/step - accuracy: 0.9989 - loss: 0

In [ ]:
y_pred = (model.predict(X_test_pad) > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 129ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       473
           1       0.99      0.97      0.98       527

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

Confusion Matrix:
[[470   3]
 [ 16 511]]


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/lstm"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# simpan model
model.save(f"{SAVE_DIR}/lstm_job_fraud.h5")

# simpan tokenizer
with open(f"{SAVE_DIR}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Model dan tokenizer berhasil disimpan ke Google Drive")
print(SAVE_DIR)


✅ Model dan tokenizer berhasil disimpan ke Google Drive
/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/lstm


# **DISTILBERT**

In [ ]:
!pip install -q transformers datasets accelerate
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
df = pd.read_csv("dataset_job_5000.csv")
df = df.fillna("")

df["text"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["company_profile"]
)

df["label"] = df["fraudulent"]

print(df.shape)
df.head()

(5000, 12)


,title,description,requirements,company_profile,location,salary_range,employment_type,industry,benefits,fraudulent,text,label
0,Health and safety adviser,Lead unit better computer assume majority. Ear...,"Basic knowledge in community, no degree requir...",Williams PLC - Established 1976.,Martinshire,$59176-$71806,Part-Time,Finance,Remote work opportunities,1,Health and safety adviser Lead unit better com...,1
1,Graduates: English Teacher Overseas,"Play with kids, get paid for it :-)Love travel...",University degree required. TEFL / TESOL / CEL...,We help teachers get safe &amp; secure jobs ab...,"ZA, WC, Cape Town",,Contract,Education Management,See job description,0,Graduates: English Teacher Overseas Play with ...,0
2,Marketing Professional,Valor Services is searching for an exceptional...,QUALIFICATIONS REQUIRED:A bachelor’s degree in...,Valor Services provides Workforce Solutions th...,"US, PA,",,Full-time,Oil & Energy,,0,Marketing Professional Valor Services is searc...,0
3,"Surveyor, building",Service social eight buy. Earn $5000/week! Imm...,"Basic knowledge in occur, no degree required. ...","Cox, Byrd and Rosario - Established 1972.",Toddview,$44289-$111142,Internship,Automotive,Sign-on bonus,1,"Surveyor, building Service social eight buy. E...",1
4,Ergonomist,Assume then late kind little. Earn $5000/week!...,"Basic knowledge in new, no degree required. Fl...",Gallegos Group - Established 1970.,West Kimberly,$32645-$100782,Part-Time,IT,Flexible hours,1,Ergonomist Assume then late kind little. Earn ...,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_fraud",
    eval_strategy="epoch",      # ⬅️ FIXED (bukan evaluation_strategy)
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    report_to="none"            # ⬅️ MATIKAN wandb (INI PENTING)
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": (preds == labels).mean()
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.087128,0.974000
2,0.087200,0.070045,0.980000
3,0.087200,0.072495,0.978000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=750, training_loss=0.07149018096923829, metrics={'train_runtime': 17133.397, 'train_samples_per_second': 0.7, 'train_steps_per_second': 0.044, 'total_flos': 794804391936000.0, 'train_loss': 0.07149018096923829, 'epoch': 3.0})

In [ ]:
pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0       0.96      1.00      0.98       473
           1       1.00      0.96      0.98       527

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

[[472   1]
 [ 19 508]]


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/distilbert"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("✅ Model & tokenizer berhasil disimpan ke Drive")
print(SAVE_DIR)

✅ Model & tokenizer berhasil disimpan ke Drive
/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/distilbert


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/distilbert"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(
    SAVE_DIR,
    safe_serialization=False   # ⬅️ INI KUNCI UTAMA
)
tokenizer.save_pretrained(SAVE_DIR)

print("✅ Model & tokenizer berhasil disimpan ke Drive")
print(SAVE_DIR)

✅ Model & tokenizer berhasil disimpan ke Drive
/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/distilbert


# **BERT**

In [ ]:
!pip install -q transformers datasets accelerate
import os
os.environ["WANDB_DISABLED"] = "true"   # pastikan wandb mati

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
df = pd.read_csv("dataset_job_5000.csv")
df = df.fillna("")

df["text"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["company_profile"]
)

df["label"] = df["fraudulent"]

print(df.shape)
df.head()

(5000, 12)


,title,description,requirements,company_profile,location,salary_range,employment_type,industry,benefits,fraudulent,text,label
0,Health and safety adviser,Lead unit better computer assume majority. Ear...,"Basic knowledge in community, no degree requir...",Williams PLC - Established 1976.,Martinshire,$59176-$71806,Part-Time,Finance,Remote work opportunities,1,Health and safety adviser Lead unit better com...,1
1,Graduates: English Teacher Overseas,"Play with kids, get paid for it :-)Love travel...",University degree required. TEFL / TESOL / CEL...,We help teachers get safe &amp; secure jobs ab...,"ZA, WC, Cape Town",,Contract,Education Management,See job description,0,Graduates: English Teacher Overseas Play with ...,0
2,Marketing Professional,Valor Services is searching for an exceptional...,QUALIFICATIONS REQUIRED:A bachelor’s degree in...,Valor Services provides Workforce Solutions th...,"US, PA,",,Full-time,Oil & Energy,,0,Marketing Professional Valor Services is searc...,0
3,"Surveyor, building",Service social eight buy. Earn $5000/week! Imm...,"Basic knowledge in occur, no degree required. ...","Cox, Byrd and Rosario - Established 1972.",Toddview,$44289-$111142,Internship,Automotive,Sign-on bonus,1,"Surveyor, building Service social eight buy. E...",1
4,Ergonomist,Assume then late kind little. Earn $5000/week!...,"Basic knowledge in new, no degree required. Fl...",Gallegos Group - Established 1970.,West Kimberly,$32645-$100782,Part-Time,IT,Flexible hours,1,Ergonomist Assume then late kind little. Earn ...,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_ds = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    auc = roc_auc_score(labels, probs)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_fraud",

    eval_strategy="epoch",      # FIX transformers terbaru
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=32,     # BERT lebih berat
    per_device_eval_batch_size=32,
    num_train_epochs=2,

    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_dir="./logs",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,No log,0.085914,0.976000,0.992172,0.962049,0.976879,0.989698
2,No log,0.081776,0.977000,0.992188,0.963947,0.977863,0.992069


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=250, training_loss=0.09087841033935547, metrics={'train_runtime': 10423.5329, 'train_samples_per_second': 0.767, 'train_steps_per_second': 0.024, 'total_flos': 526222110720000.0, 'train_loss': 0.09087841033935547, 'epoch': 2.0})

In [ ]:
pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       473
           1       0.99      0.96      0.98       527

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

Confusion Matrix:
[[469   4]
 [ 19 508]]


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/bert"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(
    SAVE_DIR,
    safe_serialization=False   # ⬅️ INI KUNCI UTAMA
)
tokenizer.save_pretrained(SAVE_DIR)

print("✅ Model & tokenizer berhasil disimpan ke Drive")
print(SAVE_DIR)

✅ Model & tokenizer berhasil disimpan ke Drive
/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/bert


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/bert"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("✅ Model & tokenizer BERT berhasil disimpan ke Drive")
print(SAVE_DIR)

✅ Model & tokenizer BERT berhasil disimpan ke Drive
/content/drive/MyDrive/Klasifikasi_Lowongan_Pekerjaan/bert
